# dscraft.tune quickstart

This notebook demonstrates `dscraft.tune`'s `ProgrammaticAdapter`: a real (tiny) LoRA fine-tuning loop -- genuine forward + backward + optimizer-step training, not a mock -- over a fully hermetic, from-scratch causal LM (`build_hermetic_causal_lm`, wired in automatically when no base model is supplied). No network access or bundled checkpoint file is required.

See the package README for how a real production base model (`openai-community/gpt2`, registered in `MODEL_ALLOWLIST`) would be wired in via the same `ProgrammaticAdapter` API.

This mirrors `packages/dscraft/examples/tune/lora_finetune_example.py` in notebook form.

Requires the `tune` extra:
```bash
pip install -e "packages/dscraft[tune]"
```

In [1]:
import tempfile
import warnings

# tqdm.auto emits a benign TqdmWarning at import time (via transformers'
# own import of tqdm.auto) when ipywidgets isn't installed -- it just means
# tqdm falls back to its plain-text progress bar; it does not affect the
# LoRA training loop demonstrated below. Suppressed narrowly (this category
# only), matching dscraft.forecast's own narrowly-scoped warning
# suppression pattern around known-benign third-party warnings.
from tqdm.std import TqdmWarning

warnings.filterwarnings("ignore", category=TqdmWarning)

# transformers logs an informational notice (`loss_type=None was set in the
# config but it is unrecognized...`) via its own logger, not via
# warnings.warn, when a config (here, the hermetic from-scratch model's
# minimal config) doesn't set `loss_type`. It's expected for this notebook's
# hermetic model and doesn't affect the real forward/backward/optimizer-step
# training below, so only this library's own logger verbosity is narrowed.
# The original verbosity level is captured here and restored in a later
# cell so it doesn't leak into any later cells/kernel session.
import transformers

original_transformers_verbosity = transformers.utils.logging.get_verbosity()
transformers.utils.logging.set_verbosity_error()

from dscraft.tune import ProgrammaticAdapter

# A tiny synthetic text corpus -- just enough repeated structure for a few
# real LoRA gradient steps to visibly move the loss on a model this small.
DATASET = [
    "the quick brown fox jumps over the lazy dog",
    "a slow green turtle naps under the warm sun",
    "the lazy dog sleeps all afternoon in the shade",
    "quick foxes and lazy dogs share the same field",
    "the warm sun helps the green turtle nap longer",
]

TRAIN_STEPS = 25

print("Preparing ProgrammaticAdapter with a hermetic from-scratch base model")
print(f"(no network access, no bundled checkpoint) over {len(DATASET)} training rows...")
adapter = ProgrammaticAdapter(learning_rate=1e-2)
adapter.prepare(None, DATASET)

n_trainable = sum(p.numel() for p in adapter.model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in adapter.model.parameters())
print(f"Base model + LoRA adapter: {n_total:,} total params, {n_trainable:,} trainable (LoRA-only).")

Preparing ProgrammaticAdapter with a hermetic from-scratch base model
(no network access, no bundled checkpoint) over 5 training rows...
Base model + LoRA adapter: 29,568 total params, 1,024 trainable (LoRA-only).


## Run real LoRA training steps

In [2]:
loss_before = adapter.compute_loss(DATASET)
print(f"Loss before fine-tuning: {loss_before:.4f}")

print(f"Running {TRAIN_STEPS} real LoRA training steps (forward + backward + optimizer step)...")
for _ in range(TRAIN_STEPS):
    result = adapter.train_step(DATASET)
print(f"Loss at final training step ({result.step}): {result.loss:.4f}")

loss_after = adapter.compute_loss(DATASET)
print(f"Loss after fine-tuning:  {loss_after:.4f}")

delta = loss_before - loss_after
direction = "decreased" if delta > 0 else "increased"
print(f"Loss {direction} by {abs(delta):.4f} over {TRAIN_STEPS} LoRA training steps.")

save_dir = tempfile.mkdtemp(prefix="dscraft_tune_lora_adapter_")
# peft's save_and_load warns (UserWarning) that it "could not find a config
# file" when saving a LoRA adapter for this notebook's hermetic
# from-scratch base model, which (deliberately, for a hermetic demo) has no
# on-disk config/tokenizer directory to look up -- peft falls back to
# assuming the vocabulary was unmodified, which is correct here. Suppressed
# narrowly, scoped only around this call, only for peft's own save path.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", category=UserWarning, module="peft.utils.save_and_load"
    )
    adapter.save_adapter(save_dir)
print(f"Saved LoRA adapter weights to: {save_dir}")

Loss before fine-tuning: 3.4821
Running 25 real LoRA training steps (forward + backward + optimizer step)...
Loss at final training step (25): 3.3238
Loss after fine-tuning:  3.2975
Loss decreased by 0.1846 over 25 LoRA training steps.
Saved LoRA adapter weights to: /var/folders/bw/9966jv5s41zgvrvx6hpss6bh0000gn/T/dscraft_tune_lora_adapter_zx8uz9v9


In [3]:
# Restore transformers' logger verbosity to what it was before this
# notebook narrowed it above, so the change doesn't leak into any later
# cells/kernel session.
transformers.utils.logging.set_verbosity(original_transformers_verbosity)
print(f"Restored transformers logging verbosity to: {original_transformers_verbosity}")

Restored transformers logging verbosity to: 30


## Summary

This notebook prepared a `ProgrammaticAdapter` over a hermetic, from-scratch causal LM, ran real LoRA fine-tuning steps (forward + backward + optimizer step) over a tiny synthetic text corpus, and confirmed the training loop's loss moved as fine-tuning steps ran, then saved the resulting LoRA adapter weights.

See `packages/dscraft/README.md`'s `## dscraft.tune` section for more detail on `BaseTrainingAdapter`, what's deferred (subprocess-isolated adapters, multi-fidelity BOHB tuning, real GGUF/MLX export).